In [1]:
import pandas as pd  
import numpy as np
import re
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LassoCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from sklearn.model_selection import train_test_split  
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from math import sqrt
from sklearn.covariance import EllipticEnvelope  
from sklearn.ensemble import RandomForestRegressor  
from sklearn.metrics import mean_absolute_error, mean_squared_error  

In [2]:
# 1。 提取梯户比
def chinese_to_arabic(cn_str):
    cn_str = str(cn_str)
    cn_num_map = {'零': 0, '一': 1, '二': 2, '两': 2, '三': 3, '四': 4, '五': 5, '六': 6, '七': 7, '八': 8, '九': 9}
    cn_unit_map = {'十': 10, '百': 100}
    try:
        return int(cn_str)
    except ValueError:
        pass
    if cn_str.startswith('十'):
        cn_str = '一' + cn_str
    res = 0
    sec_res = 0
    unit = 1
    for char in reversed(cn_str):
        if char in cn_unit_map:
            unit = cn_unit_map[char]
            if unit > sec_res:
                res += sec_res
                sec_res = 0
        elif char in cn_num_map:
            sec_res += cn_num_map[char] * unit
    res += sec_res
    return res
# 2. 定义一个包装函数来提取和计算比例    
def get_ratio(text):
    text = str(text)  
    parts = re.findall(r'(\d+|[一二三四五六七八九十百]+)', text)
    
    if len(parts) == 2: 
        try:
            
            num_elevators = chinese_to_arabic(parts[0])  # 提取电梯数
            num_households = chinese_to_arabic(parts[1])  # 提取户数
            
            if num_elevators > 0:  
                return num_households / num_elevators  
        except:  
            return np.nan  
    
    return np.nan  
def classify_common_floor(row):
    """按“当前楼层/总楼层”比例分类普通楼层"""
    if row['楼层类型'] != '普通楼层':
        return row['楼层类型']
    try:
        current_floor_map = {'低': 1, '中': 2, '高': 3, '顶': 4, '底': 0}
        current_floor = current_floor_map.get(row['初始楼层类型'], 1)  
        total_floors = row['总楼层']
        if pd.notna(total_floors) and total_floors > 0:
            ratio = current_floor / total_floors
            if ratio < 0.33:
                return '低楼层'
            elif ratio < 0.66:
                return '中楼层'
            else:
                return '高楼层'
    except (ValueError, TypeError):
        pass
    return '普通楼层'


In [3]:
# 1. 数据处理
data_rent =r"D:\人工智能\Python exam\ruc_Class25Q2_train_price.csv"
df1 = pd.read_csv(data_rent, dtype=str)
#print("===== df1 原始数据基本信息 =====")
#print(df1.info()) 
non_null_counts_df1 = df1.notnull().sum()
keep_cols_df1 = non_null_counts_df1[non_null_counts_df1 > 60000].index.tolist()
df1 = df1[keep_cols_df1]
#print("\n===== df1 筛选后数据基本信息 =====")
#print(df1.info())
#df1 = df1.dropna()  
#df1 = df1.drop_duplicates()  
#print(df1.info())
df1['交易时间'] = pd.to_datetime(df1['交易时间'], errors='coerce')
df1['交易年份'] = df1['交易时间'].dt.year

def parse_build_year(year_str):
    year_str = str(year_str)
    years = re.findall(r'\d{4}', year_str)
    if years:
        return np.mean([int(y) for y in years])
    return np.nan 
df1['建筑年份'] = df1['建筑年代'].apply(parse_build_year)
df1['房龄'] = df1['交易年份'] - df1['建筑年份']
df1['卧室数'] = df1['房屋户型'].astype(str).str.extract(r'(\d+)[室房]').fillna(0).astype(int)
df1['客厅数'] = df1['房屋户型'].astype(str).str.extract(r'(\d+)厅').fillna(0).astype(int)
df1['卫生间数'] = df1['房屋户型'].astype(str).str.extract(r'(\d+)卫').fillna(0).astype(int)
df1['配备电梯'] = df1['配备电梯'].apply(lambda x: 1 if str(x) == '有' else 0)
s = df1['所在楼层'].astype(str)
df1['总楼层'] = df1['所在楼层'].astype(str).str.extract(r'(\d+)层').squeeze()
df1['总楼层'] = pd.to_numeric(df1['总楼层'], errors='coerce')
df1['初始楼层类型'] = s.str.extract(r'^(.*?)/').squeeze()
df1['楼层类型'] = s.str.extract(r'^(低楼层|中楼层|高楼层)').squeeze()
df1['楼层类型'] = df1['楼层类型'].fillna('普通楼层')
df1['楼层类型'] = df1.apply(classify_common_floor, axis=1)
df1['梯户比例'] = df1['梯户比例'].apply(get_ratio).fillna(1)  
numeric_cols = ['Price','区域','板块', '建筑面积', 'lon', 'lat', 'coord_x', 'coord_y','城市','房屋总数','楼栋总数','绿 化 率','容 积 率','物 业 费','燃气费','停车位','停车费用']
def process_range_text(text):
    text_str = str(text)
    cleaned_text = re.sub(r"[^\d.-]", "", text_str)
    if "-" in cleaned_text:
        num1, num2 = cleaned_text.split("-")
        return (float(num1) + float(num2)) / 2
    else:
        return float(cleaned_text) if cleaned_text.strip() else np.nan

for col in numeric_cols:
    if col == 'Price':
        
        df1[col] = df1[col].str.replace('¥', '').str.replace(',', '')
        df1[col] = df1[col].apply(process_range_text)
        #df1[col] = df1[col] / 1000000
    
    elif col == '建筑面积':
        
        df1[col] = df1[col].str.replace('㎡', '')
        df1[col] = df1[col].apply(process_range_text)
    
    elif col == '房屋总数':
        
        df1[col] = df1[col].str.replace('户', '')
        df1[col] = df1[col].apply(process_range_text)
    
    elif col == '楼栋总数':
        
        df1[col] = df1[col].str.replace('栋', '')
        df1[col] = df1[col].apply(process_range_text)
    
    elif col == '绿化率':
        
        df1[col] = df1[col].str.replace('%', '')
        df1[col] = df1[col].apply(process_range_text)
        df1[col] = df1[col] / 100  
    
    else:
        
        df1[col] = df1[col].apply(process_range_text)
df1['户均楼栋房屋数'] = df1['房屋总数'] / df1['楼栋总数']
df1['每户停车位'] = df1['停车位'] / df1['房屋总数']
df1['房屋朝向'] = df1['房屋朝向'].astype(str).str.replace(' ', '') 


base_directions = ['东', '南', '西', '北']
for direction in base_directions:

    df1[f'朝向_{direction}'] = df1['房屋朝向'].str.contains(direction, na=False).astype(int)


df1['房屋朝向'] = (df1['房屋朝向'] == '未知').astype(int)

if all(col in df1.columns for col in ['核心卖点', '户型介绍', '周边配套']):
        df1['description_combined'] = (
            df1['核心卖点'].fillna('') + 
            df1['户型介绍'].fillna('') + 
            df1['周边配套'].fillna('')
        )
        

        objective_keywords = ['户型方正', '人车分流', '学区', '地铁', '医院', '商场', '超市', '公园', '菜市场']
        for keyword in objective_keywords:
            df1[f'Desc_{keyword}'] = df1['description_combined'].str.contains(keyword, na=False).astype(int)

if '客户反馈' in df1.columns:
        positive_keywords = ['体验佳', '干净', '安静', '方便', '采光好', '物业好', '安全','好', '安全', '阳光充足', '整洁']
        negative_keywords = ['老旧', '费高', '噪音', '通风差', '潮湿', '漏水', '老化', '乱', '卫生差','一般']
        
        df1['积极反馈'] = df1['客户反馈'].fillna('').apply(
            lambda x: sum(1 for word in positive_keywords if word in x)
        )
        df1['消极反馈'] = df1['客户反馈'].fillna('').apply(
            lambda x: sum(1 for word in negative_keywords if word in x)
        )
        df1['综合反馈'] = df1['积极反馈'] - df1['消极反馈']

numeric_cols = df1.select_dtypes(include=['float64', 'int64']).columns.tolist()
df1[numeric_cols] = df1[numeric_cols].fillna(df1[numeric_cols].median())
print("===== 数值列转换后信息 =====")
#print(df1.info())
#print(df1.head(15)) 

#print("\n数据统计描述：")
#print(df1.describe())  
#print("\n前5行数据：")
print(df1.head()) 

===== 数值列转换后信息 =====
    城市     区域      板块         Price      房屋户型        所在楼层    建筑面积  房屋朝向  建筑结构  \
0  0.0  109.0   150.0  6.194049e+06  2室1厅1厨1卫   中楼层 (共5层)   52.30     0  混合结构   
1  0.0   65.0   299.0  4.354153e+06  3室1厅1厨1卫    顶层 (共6层)  127.44     0  混合结构   
2  0.0   62.0   911.0  3.321992e+06  3室2厅1厨2卫   低楼层 (共6层)  118.02     0  钢混结构   
3  0.0  123.0  1102.0  7.895656e+06  6室3厅1厨3卫    底层 (共2层)  293.23     0  混合结构   
4  0.0   81.0   295.0  1.902960e+06     1房间1卫  中楼层 (共10层)   39.85     0  钢混结构   

  装修情况  ...  楼层类型    户均楼栋房屋数     每户停车位 朝向_东 朝向_南 朝向_西 朝向_北 积极反馈 消极反馈 综合反馈  
0   精装  ...   中楼层  69.315789  0.227790    0    1    0    1    0    2   -2  
1   精装  ...   低楼层  57.925000  0.668968    0    1    0    1    0    0    0  
2   简装  ...   低楼层  77.700000  0.208494    1    1    0    0    1    1    0  
3   精装  ...   中楼层   2.444444  7.575758    1    1    1    1    1    0    1  
4   精装  ...   中楼层  88.684211  1.068249    0    1    0    0    0    1   -1  

[5 rows x 63 columns]


In [4]:
# 确定特征（X）和目标变量（y）
X_rent = df1.drop(columns=['Price']) 
y_rent = df1['Price'] 
X_train_rent, X_test_rent, y_train_rent, y_test_rent = train_test_split(
    X_rent, y_rent,
    test_size=0.3,  
    random_state=111  
)
df1 = df1.loc[:, ~df1.columns.duplicated()]  

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# 1. 对数转换
df1['Price'] = np.log(df1['Price'])  
df1['建筑面积'] = np.log(df1['建筑面积'])    

numeric_cols = df1.select_dtypes(include=['float64', 'int64']).columns.tolist()

X = df1.drop('Price', axis=1)  
y = df1['Price']               
numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()

one_hot_cols = [col for col in numeric_cols if X[col].nunique() <= 2] 
continuous_cols = [col for col in numeric_cols if col not in one_hot_cols] 

preprocessor = ColumnTransformer(
    transformers=[
       
        ('num', StandardScaler(), continuous_cols),
       
        ('cat', 'passthrough', one_hot_cols)
    ],
    remainder='drop'  
)

X_processed = preprocessor.fit_transform(X) 

processed_columns = continuous_cols + one_hot_cols 
X_processed_df = pd.DataFrame(X_processed, columns=processed_columns, index=X.index)


print(X_processed_df.head())

         城市        区域        板块      建筑面积      梯户比例       lon       lat  \
0 -1.137053  1.289079 -1.356638 -1.236689  0.234847  0.451147  1.476751   
1 -1.137053 -0.016768 -0.911779  0.784225 -0.485006  0.445143  1.496152   
2 -1.137053 -0.105803  0.915431  0.609983  0.954700  0.412888  1.438494   
3 -1.137053  1.704576  1.485687  2.675039 -0.485006  0.509907  1.519242   
4 -1.137053  0.458086 -0.923721 -1.853581 -0.485006  0.435773  1.472852   

       房屋总数      楼栋总数     绿 化 率  ...     每户停车位      积极反馈      消极反馈      综合反馈  \
0 -0.313794 -0.177517 -0.036774  ... -0.225112 -0.609853  3.637521 -2.623409   
1  0.239193  0.179000 -0.036774  ... -0.030045 -0.609853 -0.530865 -0.133985   
2 -0.182736 -0.160540 -0.036774  ... -0.233644  1.073182  1.553328 -0.133985   
3 -1.005581 -0.041701  0.008098  ...  3.023791  1.073182 -0.530865  1.110728   
4 -0.110295 -0.177517  0.096509  ...  0.146497 -0.609853  1.553328 -1.378697   

   房屋朝向  配备电梯  朝向_东  朝向_南  朝向_西  朝向_北  
0   0.0   0.0   0.0   1.0   

In [6]:

if 'X_processed_df' in locals():  
    print("\n==== 预处理后特征矩阵各列缺失值统计 ====")
    processed_missing = X_processed_df.isnull().sum().reset_index()
    processed_missing.columns = ["列名", "缺失值数量"]
    processed_missing["是否存在缺失"] = processed_missing["缺失值数量"] > 0
    print(processed_missing)


==== 预处理后特征矩阵各列缺失值统计 ====
         列名  缺失值数量  是否存在缺失
0        城市      0   False
1        区域      0   False
2        板块      0   False
3      建筑面积      0   False
4      梯户比例      0   False
5       lon      0   False
6       lat      0   False
7      房屋总数      0   False
8      楼栋总数      0   False
9     绿 化 率      0   False
10    容 积 率      0   False
11    物 业 费      0   False
12      燃气费      0   False
13      停车位      0   False
14     停车费用      0   False
15  coord_x      0   False
16  coord_y      0   False
17     建筑年份      0   False
18       房龄      0   False
19      卧室数      0   False
20      客厅数      0   False
21     卫生间数      0   False
22      总楼层      0   False
23  户均楼栋房屋数      0   False
24    每户停车位      0   False
25     积极反馈      0   False
26     消极反馈      0   False
27     综合反馈      0   False
28     房屋朝向      0   False
29     配备电梯      0   False
30     朝向_东      0   False
31     朝向_南      0   False
32     朝向_西      0   False
33     朝向_北      0   False


In [7]:

X = X_processed  
y = df1['Price'].values  

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=111  
)
print(f"训练集样本数: {X_train.shape[0]}, 测试集样本数: {X_test.shape[0]}")



# 计算训练集目标变量的IQR
Q1 = np.percentile(y_train, 25)
Q3 = np.percentile(y_train, 75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 筛选训练集非异常值样本
train_mask = (y_train >= lower_bound) & (y_train <= upper_bound)
X_train_clean = X_train[train_mask]
y_train_clean = y_train[train_mask]
print(f"异常值处理前训练集样本数: {X_train.shape[0]} → 处理后: {X_train_clean.shape[0]}")



# 3.1 生成二次多项式特征（含交互项）
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_clean)  
X_test_poly = poly.transform(X_test)  

训练集样本数: 93483, 测试集样本数: 10388
异常值处理前训练集样本数: 93483 → 处理后: 92452


In [8]:

from sklearn.feature_selection import SelectFromModel
# 3.2 用Lasso做特征选择
lasso_selector = Lasso(alpha=0.01, random_state=111)
lasso_selector.fit(X_train_poly, y_train_clean)
selector = SelectFromModel(lasso_selector, prefit=True)

# 筛选后的特征
X_train_selected = selector.transform(X_train_poly)
X_test_selected = selector.transform(X_test_poly)
print(f"特征选择前维度: {X_train_poly.shape[1]} → 选择后: {X_train_selected.shape[1]}")


特征选择前维度: 629 → 选择后: 94


In [9]:
from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import SelectFromModel

original_feature_names = df1.drop('Price', axis=1).columns.tolist()
# 1. 获取多项式特征的完整列名
poly_feature_names = poly.get_feature_names_out(input_features=processed_columns)

# 2. 获取特征选择的掩码
selected_mask = selector.get_support()

# 3. 过滤得到被选中的特征名
selected_features = poly_feature_names[selected_mask]

print("\n==== Lasso选择后的特征变量 ====")
for idx, feat in enumerate(selected_features, 1):
    print(f"{idx}. {feat}")
    


==== Lasso选择后的特征变量 ====
1. 城市
2. 区域
3. 建筑面积
4. 梯户比例
5. lon
6. 房屋总数
7. 容 积 率
8. 物 业 费
9. 燃气费
10. 房龄
11. 卧室数
12. 卫生间数
13. 总楼层
14. 综合反馈
15. 配备电梯
16. 城市^2
17. 城市 lat
18. 城市 容 积 率
19. 城市 燃气费
20. 城市 停车位
21. 城市 coord_y
22. 城市 客厅数
23. 城市 总楼层
24. 城市 户均楼栋房屋数
25. 区域^2
26. 区域 板块
27. 区域 lon
28. 区域 客厅数
29. 区域 户均楼栋房屋数
30. 板块 建筑面积
31. 板块 容 积 率
32. 板块 户均楼栋房屋数
33. 建筑面积^2
34. 建筑面积 梯户比例
35. 建筑面积 容 积 率
36. 建筑面积 配备电梯
37. 梯户比例^2
38. 梯户比例 房屋总数
39. 梯户比例 建筑年份
40. 梯户比例 房龄
41. 梯户比例 户均楼栋房屋数
42. lon^2
43. lon 燃气费
44. lon coord_x
45. lon 房龄
46. lon 总楼层
47. lon 户均楼栋房屋数
48. lon 配备电梯
49. lon 朝向_南
50. lat^2
51. lat 燃气费
52. lat 房龄
53. lat 卫生间数
54. lat 户均楼栋房屋数
55. lat 朝向_南
56. lat 朝向_北
57. 房屋总数^2
58. 房屋总数 楼栋总数
59. 房屋总数 停车位
60. 房屋总数 配备电梯
61. 楼栋总数^2
62. 楼栋总数 容 积 率
63. 楼栋总数 燃气费
64. 楼栋总数 建筑年份
65. 楼栋总数 总楼层
66. 绿 化 率^2
67. 容 积 率^2
68. 容 积 率 燃气费
69. 容 积 率 停车位
70. 容 积 率 总楼层
71. 容 积 率 户均楼栋房屋数
72. 物 业 费^2
73. 物 业 费 燃气费
74. 物 业 费 建筑年份
75. 燃气费^2
76. 燃气费 房龄
77. 燃气费 配备电梯
78. 停车位^2
79. 停车位 总楼层
80. 停车费用^2
81. coord_x 总楼层
82. coord_x 户均楼

In [10]:
import joblib  


train_keep_cols = keep_cols_df1 


train_preprocessor = preprocessor  

# 3. 保存训练集的多项式特征生成器和特征选择器
train_poly = poly  
train_selector = selector  


train_numeric_cols = df1.select_dtypes(include=['float64', 'int64']).columns.tolist()
train_medians = df1[train_numeric_cols].median() 


# 1. 保存训练集筛选列
joblib.dump(train_keep_cols, "train_keep_price_cols.pkl")

# 2. 保存预处理组件
joblib.dump(train_preprocessor, "train_price_preprocessor.pkl")

# 3. 保存多项式特征生成器
joblib.dump(train_poly, "train_price_poly.pkl")

# 4. 保存特征选择器
joblib.dump(train_selector, "train_price_selector.pkl")

# 5. 保存训练集中位数
joblib.dump(train_medians, "train_price_medians.pkl")


# 6. 保存训练集最终特征列名
train_processed_columns = processed_columns  
joblib.dump(train_processed_columns, "train_processed_price_columns.pkl")


print("所有训练集配置已保存为.pkl文件！")

所有训练集配置已保存为.pkl文件！


In [11]:

models = {
    "OLS": LinearRegression(),
    "Lasso": Lasso(alpha=0.01, random_state=111),
    "Ridge": Ridge(alpha=1.0, random_state=111),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=111),
    "RandomForest": RandomForestRegressor(random_state=111)  
}


ridge_params = {"alpha": [0.001, 0.01, 0.1, 1, 10, 100]}
ridge_grid = GridSearchCV(
    models["Ridge"],  
    ridge_params,
    cv=6, 
    scoring='neg_mean_absolute_error',
    n_jobs=1  
)


ridge_grid.fit(X_train_selected, y_train_clean)
best_ridge = ridge_grid.best_estimator_


print("\nRidge Best Parameters:", ridge_grid.best_params_)


rf_model = models["RandomForest"]


rf_model.fit(X_train_selected, y_train_clean)


y_train_pred_rf = rf_model.predict(X_train_selected)
y_test_pred_rf = rf_model.predict(X_test_selected)


Ridge Best Parameters: {'alpha': 10}


In [12]:
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)  
    mse = mean_squared_error(y_true, y_pred)   
    rmse = np.sqrt(mse)                          
    return mae, rmse
rf_train_mae, rf_train_rmse = evaluate_model(y_train_clean, y_train_pred_rf)
rf_test_mae, rf_test_rmse = evaluate_model(y_test, y_test_pred_rf)


print("\n==== 随机森林模型评估结果 ====")
print(f"训练集MAE: {rf_train_mae:.4f}, 训练集RMSE: {rf_train_rmse:.4f}")
print(f"测试集MAE: {rf_test_mae:.4f}, 测试集RMSE: {rf_test_rmse:.4f}")


rf_results = {
    "In-sample MAE": rf_train_mae,       
    "In-sample RMAE": rf_train_rmse,     
    "Out-of-sample MAE": rf_test_mae,    
    "Out-of-sample RMAE": rf_test_rmse   
   
}


joblib.dump(rf_model, "trained_rf_model.pkl")  # 保存模型到本地

print("随机森林模型已保存为：trained_rf_model.pkl")


==== 随机森林模型评估结果 ====
训练集MAE: 0.0340, 训练集RMSE: 0.0491
测试集MAE: 0.0961, 测试集RMSE: 0.1452
随机森林模型已保存为：trained_rf_model.pkl


In [13]:


def evaluate_model(model, X_train, y_train, X_test, y_test):
    """计算模型的MAE/RMAE（样本内/外+交叉验证）"""
    
    y_pred_train = model.predict(X_train)
    mae_train = mean_absolute_error(y_train, y_pred_train)
    rmae_train = np.sqrt(mae_train)
    
  
    y_pred_test = model.predict(X_test)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    rmae_test = np.sqrt(mae_test)
    
    
    cv_scores = cross_val_score(
        model, X_train, y_train, cv=6, scoring="neg_mean_absolute_error"
    )
    mae_cv = -cv_scores.mean()
    rmae_cv = np.sqrt(mae_cv)
    
    return {
        "In-sample MAE": mae_train,
        "In-sample RMAE": rmae_train,
        "Out-of-sample MAE": mae_test,
        "Out-of-sample RMAE": rmae_test,
        "6-fold CV MAE": mae_cv,
        "6-fold CV RMAE": rmae_cv
    }


ridge_results = evaluate_model(best_ridge, X_train_selected, y_train_clean, X_test_selected, y_test)
print("\nRidge模型性能:")
for metric, val in ridge_results.items():
    print(f"{metric}: {val:.4f}")



# 训练OLS模型
ols = LinearRegression()
ols.fit(X_train_selected, y_train_clean)
ols_results = evaluate_model(ols, X_train_selected, y_train_clean, X_test_selected, y_test)

print("\nOLS模型性能:")
for metric, val in ols_results.items():
    print(f"{metric}: {val:.4f}")


elastic_net = ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=111)
elastic_net.fit(X_train_selected, y_train_clean)


elastic_results = evaluate_model(elastic_net, X_train_selected, y_train_clean, X_test_selected, y_test)


print("\nElasticNet模型性能:")
for metric, val in elastic_results.items():
    print(f"{metric}: {val:.4f}")

all_results = {
    "OLS": ols_results,
    "Ridge（最优）": ridge_results,
    "ElasticNet": elastic_results,  
    "随机森林": rf_results  
}

results_df = pd.DataFrame(all_results).T  
print("\n所有模型性能对比:")
print(results_df.round(4))  



print(f"\n异常值处理后训练集样本数：{X_train_clean.shape[0]}")



Ridge模型性能:
In-sample MAE: 0.3211
In-sample RMAE: 0.5666
Out-of-sample MAE: 0.3220
Out-of-sample RMAE: 0.5674
6-fold CV MAE: 0.3216
6-fold CV RMAE: 0.5671

OLS模型性能:
In-sample MAE: 0.3211
In-sample RMAE: 0.5666
Out-of-sample MAE: 0.3220
Out-of-sample RMAE: 0.5674
6-fold CV MAE: 0.3216
6-fold CV RMAE: 0.5671

ElasticNet模型性能:
In-sample MAE: 0.3269
In-sample RMAE: 0.5718
Out-of-sample MAE: 0.3279
Out-of-sample RMAE: 0.5726
6-fold CV MAE: 0.3274
6-fold CV RMAE: 0.5722

所有模型性能对比:
            In-sample MAE  In-sample RMAE  Out-of-sample MAE  \
OLS                0.3211          0.5666             0.3220   
Ridge（最优）          0.3211          0.5666             0.3220   
ElasticNet         0.3269          0.5718             0.3279   
随机森林               0.0340          0.0491             0.0961   

            Out-of-sample RMAE  6-fold CV MAE  6-fold CV RMAE  
OLS                     0.5674         0.3216          0.5671  
Ridge（最优）               0.5674         0.3216          0.5671  
ElasticN

In [20]:

import pandas as pd
import numpy as np
import re
import joblib
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.compose import ColumnTransformer

# 加载训练集保存的关键配置
train_keep_cols = joblib.load("train_keep_price_cols.pkl")  
train_preprocessor = joblib.load("train_price_preprocessor.pkl")  
train_poly = joblib.load("train_price_poly.pkl")
train_selector = joblib.load("train_price_selector.pkl")
train_medians = joblib.load("train_price_medians.pkl")
train_processed_columns = joblib.load("train_processed_price_columns.pkl")

# 加载训练好的随机森林模型
rf_model = joblib.load("trained_rf_model.pkl")

In [21]:
# 1. 数据处理
data_rent =r"D:\人工智能\Python exam\ruc_Class25Q2_test_price.csv"
df1 = pd.read_csv(data_rent, dtype=str)
#print("===== df1 原始数据基本信息 =====")
#print(df1.info()) 
non_null_counts_df1 = df1.notnull().sum()
keep_cols_df1 = non_null_counts_df1[non_null_counts_df1 > 8000].index.tolist()
df1 = df1[keep_cols_df1]
#print("\n===== df1 筛选后数据基本信息 =====")
#print(df1.info())
#df1 = df1.dropna()  
#df1 = df1.drop_duplicates()  
#print(df1.info())

df1['交易时间'] = pd.to_datetime(df1['交易时间'], errors='coerce')
df1['交易年份'] = df1['交易时间'].dt.year

def parse_build_year(year_str):
    year_str = str(year_str)
    years = re.findall(r'\d{4}', year_str)
    
    if years:
        return np.mean([int(y) for y in years])
    return np.nan 


df1['建筑年份'] = df1['建筑年代'].apply(parse_build_year)

df1['房龄'] = df1['交易年份'] - df1['建筑年份']

df1['卧室数'] = df1['房屋户型'].astype(str).str.extract(r'(\d+)[室房]').fillna(0).astype(int)
df1['客厅数'] = df1['房屋户型'].astype(str).str.extract(r'(\d+)厅').fillna(0).astype(int)
df1['卫生间数'] = df1['房屋户型'].astype(str).str.extract(r'(\d+)卫').fillna(0).astype(int)
df1['配备电梯'] = df1['配备电梯'].apply(lambda x: 1 if str(x) == '有' else 0)
s = df1['所在楼层'].astype(str)
df1['总楼层'] = df1['所在楼层'].astype(str).str.extract(r'(\d+)层').squeeze()
df1['总楼层'] = pd.to_numeric(df1['总楼层'], errors='coerce')
df1['初始楼层类型'] = s.str.extract(r'^(.*?)/').squeeze()
df1['楼层类型'] = s.str.extract(r'^(低楼层|中楼层|高楼层)').squeeze()
df1['楼层类型'] = df1['楼层类型'].fillna('普通楼层')
df1['楼层类型'] = df1.apply(classify_common_floor, axis=1)
df1['梯户比例'] = df1['梯户比例'].apply(get_ratio).fillna(1)  # 空值填充为1（默认1梯1户）
numeric_cols = ['区域','板块', '建筑面积', 'lon', 'lat', 'coord_x', 'coord_y','城市','房屋总数','楼栋总数','绿 化 率','容 积 率','物 业 费','燃气费','停车位','停车费用']

def process_range_text(text):
    
    text_str = str(text)
    cleaned_text = re.sub(r"[^\d.-]", "", text_str)
    
    
    if "-" in cleaned_text:
        
        num1, num2 = cleaned_text.split("-")
        return (float(num1) + float(num2)) / 2
    
    else:
        
        return float(cleaned_text) if cleaned_text.strip() else np.nan

for col in numeric_cols:
    
    if col == '建筑面积':
        
        df1[col] = df1[col].str.replace('㎡', '')
        df1[col] = df1[col].apply(process_range_text)
    
    elif col == '房屋总数':
        
        df1[col] = df1[col].str.replace('户', '')
        df1[col] = df1[col].apply(process_range_text)
    
    elif col == '楼栋总数':
        
        df1[col] = df1[col].str.replace('栋', '')
        df1[col] = df1[col].apply(process_range_text)
    
    elif col == '绿化率':
       
        df1[col] = df1[col].str.replace('%', '')
        df1[col] = df1[col].apply(process_range_text)
        df1[col] = df1[col] / 100  
    
    else:
        
        df1[col] = df1[col].apply(process_range_text)
df1['户均楼栋房屋数'] = df1['房屋总数'] / df1['楼栋总数']
df1['每户停车位'] = df1['停车位'] / df1['房屋总数']

df1['房屋朝向'] = df1['房屋朝向'].astype(str).str.replace(' ', '')  


base_directions = ['东', '南', '西', '北']
for direction in base_directions:

    df1[f'朝向_{direction}'] = df1['房屋朝向'].str.contains(direction, na=False).astype(int)


df1['房屋朝向'] = (df1['房屋朝向'] == '未知').astype(int)

if all(col in df1.columns for col in ['核心卖点', '户型介绍', '周边配套']):
        df1['description_combined'] = (
            df1['核心卖点'].fillna('') + 
            df1['户型介绍'].fillna('') + 
            df1['周边配套'].fillna('')
        )
        objective_keywords = ['户型方正', '人车分流', '学区', '地铁', '医院', '商场', '超市', '公园', '菜市场']
        for keyword in objective_keywords:
            df1[f'Desc_{keyword}'] = df1['description_combined'].str.contains(keyword, na=False).astype(int)
if '客户反馈' in df1.columns:
        positive_keywords = ['体验佳', '干净', '安静', '方便', '采光好', '物业好', '安全','好', '安全', '阳光充足', '整洁']
        negative_keywords = ['老旧', '费高', '噪音', '通风差', '潮湿', '漏水', '老化', '乱', '卫生差','一般']
        
        df1['积极反馈'] = df1['客户反馈'].fillna('').apply(
            lambda x: sum(1 for word in positive_keywords if word in x)
        )
        df1['消极反馈'] = df1['客户反馈'].fillna('').apply(
            lambda x: sum(1 for word in negative_keywords if word in x)
        )
        df1['综合反馈'] = df1['积极反馈'] - df1['消极反馈']

numeric_cols = df1.select_dtypes(include=['float64', 'int64']).columns.tolist()
df1[numeric_cols] = df1[numeric_cols].fillna(df1[numeric_cols].median())
print("===== 数值列转换后信息 =====")
print(df1.info())
#print(df1.head(15)) 

#print("\n数据统计描述：")
#print(df1.describe())  
#print("\n前5行数据：")
print(df1.head()) 

===== 数值列转换后信息 =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34017 entries, 0 to 34016
Data columns (total 81 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   ID                    34017 non-null  object        
 1   城市                    34017 non-null  float64       
 2   区域                    34017 non-null  float64       
 3   板块                    34017 non-null  float64       
 4   环线                    15677 non-null  object        
 5   房屋户型                  34003 non-null  object        
 6   所在楼层                  34017 non-null  object        
 7   建筑面积                  34017 non-null  float64       
 8   套内面积                  9915 non-null   object        
 9   房屋朝向                  34017 non-null  int64         
 10  建筑结构                  34003 non-null  object        
 11  装修情况                  34003 non-null  object        
 12  梯户比例                  34017 non-null  float64       


In [22]:
df_test = df1  
df_test['建筑面积'] = np.log(df_test['建筑面积'].astype(float) + 1e-6)

In [23]:

valid_processed_columns = [
    col for col in train_processed_columns  
    if col in df_test.columns and col != "Price"  
]


df_test_features = df_test[valid_processed_columns].copy()
print(f"测试集对齐后特征列：{df_test_features.columns.tolist()}")
print(f"测试集对齐后形状：{df_test_features.shape}")  

测试集对齐后特征列：['城市', '区域', '板块', '建筑面积', '梯户比例', 'lon', 'lat', '房屋总数', '楼栋总数', '绿 化 率', '容 积 率', '物 业 费', '燃气费', '停车位', '停车费用', 'coord_x', 'coord_y', '建筑年份', '房龄', '卧室数', '客厅数', '卫生间数', '总楼层', '户均楼栋房屋数', '每户停车位', '积极反馈', '消极反馈', '综合反馈', '房屋朝向', '配备电梯', '朝向_东', '朝向_南', '朝向_西', '朝向_北']
测试集对齐后形状：(34017, 34)


In [24]:

X_test_processed = train_preprocessor.transform(df_test_features)
print(f"测试集预处理后形状：{X_test_processed.shape}")  

测试集预处理后形状：(34017, 34)


In [25]:
# 1. 生成二次多项式特征
X_test_poly = train_poly.transform(X_test_processed)
print(f"测试集多项式特征后形状：{X_test_poly.shape}")  

# 2. 特征选择
X_test_selected = train_selector.transform(X_test_poly)
print(f"测试集特征选择后形状：{X_test_selected.shape}")  

测试集多项式特征后形状：(34017, 629)
测试集特征选择后形状：(34017, 94)


In [29]:
# 模型预测
#y_test_pred_log = rf_model.predict(X_test_selected)
y_test_pred_log = ols.predict(X_test_selected) 
print(f"预测的log_Price前5个值：{y_test_pred_log[:5]}")

预测的log_Price前5个值：[16.36462501 14.75796546 16.14975941 14.78722194 15.41218385]


In [30]:

#y_test_pred = np.exp(y_test_pred_log) * 1000000
y_test_pred = np.exp(y_test_pred_log) 



submission_df = pd.DataFrame({
    "ID": df_test["ID"], 
    "预测租金（元）": y_test_pred.astype(int)  
})


submission_df.to_csv("test_rent_price_prediction2.csv", index=False, encoding="utf-8-sig")
print("预测结果已保存为：test_rent_price_prediction2.csv")


print("\n测试集前10条预测结果：")
print(submission_df.head(10))

预测结果已保存为：test_rent_price_prediction2.csv

测试集前10条预测结果：
        ID   预测租金（元）
0  1000000  12795767
1  1000001   2566273
2  1000002  10321703
3  1000003   2642462
4  1000004   4936582
5  1000005   3012194
6  1000006   6709387
7  1000007   1942828
8  1000008   2516790
9  1000009   6073170


In [31]:
#合并
df1 = pd.read_csv(r"D:\人工智能\lecture-python-programming.notebooks-main\test_rent_price_prediction.csv")  
df2 = pd.read_csv(r"D:\人工智能\lecture-python-programming.notebooks-main\test_rent_price_prediction2.csv")  
extracted_df1 = df1[['ID', '预测租金（元）']].copy()
extracted_df2 = df2[['ID', '预测租金（元）']].copy()
combined_df = pd.concat([extracted_df2, extracted_df1], axis=0, ignore_index=True)
print("提取的两列数据：")
print(combined_df.head())  
combined_df.to_csv("合并预测结果.csv", index=False)  

提取的两列数据：
        ID   预测租金（元）
0  1000000  12795767
1  1000001   2566273
2  1000002  10321703
3  1000003   2642462
4  1000004   4936582


In [ ]:
#rent的预测代码在git上，学号2023201776 finance